# 01 — Exploration

Match **Arch Linux** packages against **MSYS2** and **MINGW**.

MSYS2 and MINGW are not designed to correspond to each other — they are two
separate porting efforts of the same upstream software — so pairing them
directly is meaningless. Arch Linux is the neutral reference point: it packages
much of the same upstream software under close-to-upstream names.

Three inputs:

| file | contents |
| --- | --- |
| `data/arch-pkgs.jsonl` | scraped Arch/AUR packages (run `tools/scrape_arch.py`) |
| `data/msys2-pkgs.tsv` | `ls -la` snapshot of the MSYS2 repository |
| `data/mingw-pkgs.tsv` | `ls -la` snapshot of the MINGW repository |

If you have not run the scraper yet, this notebook fetches `core` live.

In [11]:
Path.cwd().name

'notebooks'

In [14]:
import sys
from pathlib import Path

def cwd_finder(name : str, start : Path = Path.cwd()):
    start = start.absolute()
    if name in set(x.name for x in start.iterdir()):
        return start / name
    elif start == start.parent:
        return None
    else:
        return cwd_finder(name=name, start=start.parent)

src_path = cwd_finder("src")
display(src_path)
# Keep the src-layout importable when running from the repository root.
sys.path.insert(0, src_path)

from msys2_dataset.matching import match_ecosystems, summarise_matches
from msys2_dataset.packages import parse_ls_listing_file

PosixPath('/workspaces/msys2-dataset/src')

PosixPath('/workspaces/msys2-dataset/notebooks')

### Load the three ecosystems

In [15]:
DATA = cwd_finder("data")
msys2 = parse_ls_listing_file(DATA / "msys2-pkgs.tsv")
mingw = parse_ls_listing_file(DATA / "mingw-pkgs.tsv")
len(msys2), len(mingw)

(608, 3361)

The listings are `ls -la` output, so only directories are packages; `total`, `LICENSE` and `README.md` lines are metadata.

In [16]:
assert not {"total", "LICENSE", "README.md"} & {e.name for e in msys2}
assert not {"total", "LICENSE", "README.md"} & {e.name for e in mingw}
[e.name for e in msys2[:5]], [e.name for e in mingw[:3]]

(['ack', 'ansible', 'ansible-core', 'apr', 'apr-util'],
 ['mingw-w64-3proxy', 'mingw-w64-4th', 'mingw-w64-4ti2'])

### Load the scraped Arch packages

Prefer the cached scrape; fall back to fetching `core` live so the notebook
works on a fresh checkout.

In [17]:
import json

arch_path = DATA / "arch-pkgs.jsonl"
if arch_path.is_file():
    from msys2_dataset.arch import ArchPackage

    arch = [
        ArchPackage(**json.loads(line))
        for line in arch_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    source = "cached scrape"
else:
    from msys2_dataset.arch import fetch_official_repo

    arch = fetch_official_repo("core")
    source = "live core only"

from collections import Counter

print(f"{len(arch)} arch packages ({source})")
Counter(p.repo for p in arch).most_common()

FileNotFoundError: [Errno 2] No such file or directory: 'node'

### Match by normalised upstream name

`mingw-w64-ucrt-x86_64-zlib` normalises to `zlib`, which matches Arch's `zlib`.
A small alias table covers genuinely divergent names (Arch `freetype2` ↔ MSYS2
`freetype`).

In [ ]:
matches = match_ecosystems(arch, msys2, mingw)
stats = summarise_matches(matches)
stats

### Projects present in all three ecosystems

These are the genuinely portable upstream projects.

In [ ]:
all_three = [m for m in matches if m.in_arch and m.in_msys2 and m.in_mingw]
for m in all_three[:15]:
    print(f"{m.name:20} arch={m.arch[0].version:16} msys2={m.msys2[0].name:20} mingw={m.mingw[0].name}")
len(all_three)

### Where the ecosystems diverge

Arch packages a lot that MSYS2/MINGW do not: MSYS2 and MINGW are
POSIX-on-Windows toolchains, so they cover the portable core but not Arch's
Linux-only surface.

In [ ]:
arch_only = [m.name for m in matches if m.in_arch and not m.in_msys2 and not m.in_mingw]
print(f"arch only ({len(arch_only)}):", arch_only[:12])

# Projects MSYS2 ships but Arch does not, under the same normalised name.
msys2_only = [m.name for m in matches if m.in_msys2 and not m.in_arch]
print(f"msys2 only ({len(msys2_only)}):", msys2_only[:12])

### Coverage of Arch by the Windows toolchains

In [ ]:
{
    "arch coverage by msys2+mingw": round(stats["arch_matched"] / stats["arch_total"], 3),
    "arch coverage by msys2": round(stats["arch_msys2"] / stats["arch_total"], 3),
    "arch coverage by mingw": round(stats["arch_mingw"] / stats["arch_total"], 3),
}